### **Считайте датасет из файла train.csv (это данные о выживаемости на Титанике)**

In [ ]:
# Скачать файл из Google Drive
! gdown --id 1YsLUVR9bc2WQRIVC0Os1TCyx-yTwQRHq

/usr/local/lib/python3.10/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1YsLUVR9bc2WQRIVC0Os1TCyx-yTwQRHq
To: /content/train.csv
100% 61.2k/61.2k [00:00<00:00, 16.2MB/s]


In [ ]:
# Импорт модулей
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/train.csv')
df[:4]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S


### **Выберите и обоснуйте метрику для измерения качества (accuracy/precision/recall/f1-score/fbeta-score/roc-auc и т.д.). В рамках данного пункта необходимо подобрать наиболее релевантную метрику или набор метрик для вашей задачи, написав краткое обоснование (1-2 предложения)**

Наверное, классы несбалансированы, потому что большинство умерло как мы знаем из задания 4. Можно всегда предсказывать, что умер и угадывать в 61% случаев.
Если классы не сбалансированы, то здесь подойдет f1-score, потому что является средним для precision и recall, которые чувствительны к неравномерным классам.

### **Постройте бейзлайн и ML-модель классификации (LogisticRegression или любая другая, которая вам кажется подходящей) и оцените их качество с помощью выбранной метрики**

In [ ]:
# В каких столбцах пустые значения
df.isna().sum()[df.isna().sum() > 0]

,0
Age,177
Cabin,687
Embarked,2


In [ ]:
# Заполнить пропуски числовых столбцов
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

In [ ]:
# Числовые значения для категорий
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)  # Кодирование категориальных переменных
df[:4]

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,NaN,True,False,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,C85,False,False,False
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,NaN,False,False,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,C123,False,False,True


In [ ]:
# Делить на обучающую и текстовую выборку
from sklearn.model_selection import train_test_split

# На X нужно обучаться
X = df.drop(['Survived', 'Name', 'Ticket', 'Cabin'], axis=1)  # Удаление ненужных признаков, которые мешают предсказывать
# Нужно предсказать y
y = df['Survived']

# Размер тестовой выборки - 0.2
# Вот здесь зафиксирован random_state
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Бэйзлайн
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

# Модель на основе большинства
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train, y_train)
y_pred_dummy = dummy_clf.predict(X_test)

baseline_f1 = f1_score(y_test, y_pred_dummy)
print(f'F1-score бэйзлайна: {baseline_f1}')

F1-score бэйзлайна: 0.0


In [ ]:
# Построение модели ML
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

model_f1 = f1_score(y_test, y_pred)
print(f'F1-score модели: {model_f1}')

F1-score модели: 0.7586206896551724
